# LYD Parallel Market -- 24-48h Exchange Rate Forecast

Real data pipeline, start to finish: parse your exported WhatsApp chats,
understand the news with an LLM, engineer features, train per-currency models,
backtest, then manually query a prediction with a confidence rating.

Data sources (real, as of this version):
1. rates.txt -- exported chat from a rates-posting channel (sample: 24 posts,
   Aug 13-18 2026, covering USD, EUR, GBP, and a Sukuk instrument, all vs LYD)
2. news.txt -- exported chat from a news channel (sample: 40 posts, Aug 16-18 2026)

Read this first -- an honest finding from actually parsing your sample data:
news.txt's content (Al Ekhbariya-style Saudi/Gulf breaking news) contains zero
mentions of Libya, the dinar, or currency terms anywhere in the 40 sample
messages. It's general regional political news, not a Libya/LYD-focused feed.
The notebook is built to handle this honestly: Section 3 uses an LLM specifically
to score each item's relevance to the LYD before scoring sentiment, so an
irrelevant news source correctly produces near-zero-weight features instead of
injecting false signal. If you swap in a Libya-specific economic news source
later, this same mechanism will pick up real signal from it with no code changes.

Also honest: your sample rates data spans only ~5 days. After the feature
pipeline (which needs up to 48h of lag history and a 48h-ahead target), that
leaves roughly 25 usable training rows per currency -- enough to prove the
pipeline works end-to-end, not enough to trust the forecasts for real decisions
yet. The notebook prints explicit data-sufficiency warnings wherever this
matters, rather than presenting small-sample results as reliable.

Why LightGBM + engineered features, not a big neural model: parallel-market LYD
rate reports are sparse and irregular, not a clean tick feed. Gradient-boosted
trees on lag/rolling/news features are far more data-efficient than an
LSTM/Transformer here, retrain fast, and give feature importances you can
sanity-check.

Notebook sections: 1 setup, 2 parse real chat exports, 3 LLM news understanding,
4 (optional) live WhatsApp ingestion, 5-6 dataset finalization, 7-9 feature
pipeline, 10 training (all pairs), 11 testing/backtest, 12 manual test, 13-15
save/serve/TODO.


## 1. Setup

In [1]:
!pip install -q lightgbm anthropic 2>/dev/null
import numpy as np
import pandas as pd
import re
import sys
import os
import json
import hashlib
import joblib
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

pd.set_option("display.width", 120)
RNG = np.random.default_rng(42)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.4/95.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 3.4 MB/s eta 0:00:00


## 2. Parse real WhatsApp chat exports (primary data source)

This is the recommended way to get real data in: WhatsApp's own Export Chat
feature (per-chat menu -> More -> Export Chat -> Without Media) produces a .txt
file in a fixed, well-documented format. No QR codes, no live network
dependency, no unofficial-library risk -- just a text file. Section 4 (optional)
covers live ingestion for keeping this topped up going forward, but this is the
reliable bootstrap.

Format handled: `[H:MM AM/PM, M/D/YYYY] Sender: message text` (message text may
span multiple following lines until the next [...] header). Verified against
your actual uploaded files -- 24/24 rate messages and 40/40 news messages
parsed correctly, 0 skipped.


In [2]:
def normalize_arabic(text):
    """Strip Arabic diacritics (tashkeel) and tatweel (the elongation
    character) so spelling variants match the same pattern - found necessary
    by testing against the real file (the Sukuk line uses a stylized spelling
    with a tatweel character)."""
    if not isinstance(text, str):
        return text
    return re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)

MSG_HEADER_RE = re.compile(
    r"^\[(\d{1,2}:\d{2}\s?[AP]M),\s*(\d{1,2}/\d{1,2}/\d{4})\]\s*([^:]+):\s?(.*)$"
)

def parse_whatsapp_export(path):
    """Parses a WhatsApp 'Export Chat' .txt file into a list of
    {timestamp, sender, text} dicts. Handles multi-line messages (the export
    format only puts a new [timestamp] header at the start of each NEW
    message, so continuation lines have no marker of their own)."""
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    messages = []
    current = None
    skipped = 0
    for line in raw.splitlines():
        m = MSG_HEADER_RE.match(line.strip())
        if m:
            if current:
                messages.append(current)
            time_str, date_str, sender, first_text = m.groups()
            try:
                dt = pd.Timestamp(f"{date_str} {time_str}")
            except ValueError:
                skipped += 1
                current = None
                continue
            current = {"timestamp": dt, "sender": sender.strip(), "text": first_text}
        else:
            if current is not None and line.strip():
                current["text"] += "\n" + line
    if current:
        messages.append(current)
    print(f"{path}: parsed {len(messages)} messages, skipped {skipped} malformed header lines")
    return messages

# Currency line patterns, matched against normalized (diacritic-stripped) text.
# >>> ADJUST / extend if your channel reports different currencies or wording.
CUR_PATTERNS = {
    "USD/LYD": r"دولار[^=\n]*=\s*([\d.]+)\s*دينار\s*(🔺|🔻|➖)?\s*(⚡)?",
    "EUR/LYD": r"يورو[^=\n]*=\s*([\d.]+)\s*دينار\s*(🔺|🔻|➖)?\s*(⚡)?",
    "GBP/LYD": r"باوند[^=\n]*=\s*([\d.]+)\s*دينار\s*(🔺|🔻|➖)?\s*(⚡)?",
    "SUKUK/LYD": r"صكوك[^=\n]*=\s*([\d.]+)\s*دينار\s*(🔺|🔻|➖)?\s*(⚡)?",
}
DIRECTION_MAP = {"🔺": 1, "🔻": -1, "➖": 0}

def rates_messages_to_df(messages):
    rows = []
    for msg in messages:
        body = normalize_arabic(msg["text"])
        for pair, pattern in CUR_PATTERNS.items():
            m = re.search(pattern, body)
            if m:
                rows.append({
                    "timestamp": msg["timestamp"],
                    "pair": pair,
                    "sell_rate": float(m.group(1)),
                    "direction": DIRECTION_MAP.get(m.group(2), 0),  # channel's own self-reported move
                    "sharp_move": m.group(3) is not None,            # the lightning-bolt emphasis marker
                    "source_group": msg["sender"],
                })
    if not rows:
        return pd.DataFrame(columns=["timestamp", "pair", "sell_rate", "direction", "sharp_move", "source_group"])
    return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)

def news_messages_to_df(messages):
    if not messages:
        return pd.DataFrame(columns=["timestamp", "source_group", "text"])
    return pd.DataFrame([
        {"timestamp": m["timestamp"], "source_group": m["sender"], "text": m["text"]}
        for m in messages
    ]).sort_values("timestamp").reset_index(drop=True)


In [18]:
# >>> REPLACE these paths if your files are named/located differently.
# Checks the notebook's own directory first, then common upload locations.
MIN_ROWS_TO_USE_REAL_DATA = 20  # >>> lower/raise this bar as you see fit - also gates whether Section 4 (live ingestion) bothers running
def find_file(filename):
    candidates = [Path(filename), Path("/kaggle/input/datasets/ferasshita/datanews") / filename]
    if Path("/kaggle/input").exists():
        candidates += list(Path("/kaggle/input").glob(f"*/{filename}"))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

RATES_TXT_PATH = find_file("rates.txt")
NEWS_TXT_PATH = find_file("news.txt")

if RATES_TXT_PATH and NEWS_TXT_PATH:
    print(f"Found {RATES_TXT_PATH} and {NEWS_TXT_PATH}")
    rates_real_df = rates_messages_to_df(parse_whatsapp_export(RATES_TXT_PATH))
    news_real_df = news_messages_to_df(parse_whatsapp_export(NEWS_TXT_PATH))
    print(f"\nrates_real_df: {len(rates_real_df)} rows across pairs:")
    print(rates_real_df["pair"].value_counts())
    if len(news_real_df):
        print(f"\nnews_real_df: {len(news_real_df)} rows, "
              f"{news_real_df.timestamp.min()} -> {news_real_df.timestamp.max()}")
    else:
        print("\nnews_real_df: empty")
else:
    print("rates.txt / news.txt not found (checked working dir, /mnt/user-data/uploads, "
          "/kaggle/input) - place them next to this notebook, or set RATES_TXT_PATH / "
          "NEWS_TXT_PATH manually. Falling back to synthetic data for now (see Section 6).")
    rates_real_df, news_real_df = pd.DataFrame(), pd.DataFrame()


Found /kaggle/input/datasets/ferasshita/datanews/rates.txt and /kaggle/input/datasets/ferasshita/datanews/news.txt
/kaggle/input/datasets/ferasshita/datanews/rates.txt: parsed 24 messages, skipped 0 malformed header lines
/kaggle/input/datasets/ferasshita/datanews/news.txt: parsed 40 messages, skipped 0 malformed header lines

rates_real_df: 74 rows across pairs:
pair
USD/LYD      24
SUKUK/LYD    19
EUR/LYD      18
GBP/LYD      13
Name: count, dtype: int64

news_real_df: 40 rows, 2026-08-16 19:40:00 -> 2026-08-18 22:18:00


## 3. Generative AI news understanding

Uses Claude to read each news item and score two things separately, which
matters a lot given what Section 2 found:

- relevance (0-1): how relevant this item is to Libya's economy, currency,
  politics, or regional stability that could plausibly move the LYD
- sentiment (-1 to +1): only meaningful when relevance is non-trivial;
  negative = plausibly weakens the dinar, positive = plausibly strengthens it

Scoring relevance separately from sentiment is the important design choice
here: a naive keyword-sentiment pass would either force a sentiment score onto
totally unrelated news (false signal) or find nothing and silently contribute
zero variance. Explicit relevance scoring lets the feature pipeline correctly
down-weight an irrelevant news source instead of guessing.

Requires your own Anthropic API key (`ANTHROPIC_API_KEY` environment
variable). Without one, this falls back to a much cruder keyword heuristic --
still functional, just far less able to tell relevant from irrelevant content
the way this sample data needs. Results are cached to llm_news_cache.json so
re-running the notebook doesn't re-spend API calls on messages already scored.


In [19]:
LLM_MODEL = "claude-haiku-4-5-20251001"  # fast/cheap, appropriate for short classification calls - swap for a stronger model if you want deeper reasoning
LLM_CACHE_PATH = Path("llm_news_cache.json")

def get_anthropic_client():
    try:
        import anthropic
    except ImportError:
        print("`anthropic` package not available - falling back to keyword heuristic.")
        return None
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ANTHROPIC_API_KEY not set - falling back to keyword heuristic. "
              "Set the env var and re-run this cell to enable LLM scoring.")
        return None
    return anthropic.Anthropic(api_key=api_key)

def load_llm_cache():
    if LLM_CACHE_PATH.exists():
        try:
            return json.loads(LLM_CACHE_PATH.read_text())
        except json.JSONDecodeError:
            return {}
    return {}

def save_llm_cache(cache):
    LLM_CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False, indent=2))

NEWS_PROMPT_TEMPLATE = """You are analyzing news for its likely impact on the Libyan Dinar (LYD) parallel-market exchange rate against USD/EUR.

Read the news item below (likely Arabic) and respond with ONLY a JSON object, no other text, no markdown fences:
{{"relevance": <0.0-1.0, how relevant to Libya's economy, currency, politics, conflict, or regional stability that could plausibly affect the LYD - most general regional news unrelated to Libya should score well under 0.3>, "sentiment": <-1.0 to 1.0, only meaningful if relevance > 0.2; negative = plausibly weakens the dinar, positive = plausibly strengthens it, 0 if not applicable>, "reasoning": "<one short sentence in English>"}}

News item: {text}"""

def analyze_news_with_llm(client, text, cache):
    key = hashlib.sha256(text.encode("utf-8")).hexdigest()
    if key in cache:
        return cache[key]
    try:
        resp = client.messages.create(
            model=LLM_MODEL,
            max_tokens=200,
            messages=[{"role": "user", "content": NEWS_PROMPT_TEMPLATE.format(text=text)}],
        )
        raw = resp.content[0].text.strip()
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
        result = json.loads(raw)
        result = {
            "relevance": float(result.get("relevance", 0.0)),
            "sentiment": float(result.get("sentiment", 0.0)),
            "reasoning": str(result.get("reasoning", "")),
        }
    except Exception as e:
        result = {"relevance": None, "sentiment": None, "reasoning": f"LLM call failed: {e}"}
    cache[key] = result
    return result


Keyword fallback (used automatically with no API key, and as a per-item
fallback if an individual LLM call fails). Deliberately crude compared to the
LLM path -- it can only recognize explicit Libya/currency keywords and simple
up/down Arabic words, so it will correctly find ~nothing in a feed like your
sample news.txt rather than fabricate signal.


In [20]:
UP_WORDS = ["ارتفاع", "زيادة", "قفز", "صعود", "غلا"]
DOWN_WORDS = ["انخفاض", "تراجع", "هبوط", "نزول", "رخص"]
RELEVANCE_KEYWORDS = ["ليبيا", "دينار", "الدولار", "اقتصاد", "عملة", "مصرف", "السوق الموازي"]

def keyword_relevance_sentiment(text):
    text_n = normalize_arabic(text) if isinstance(text, str) else ""
    relevance = 0.9 if any(k in text_n for k in RELEVANCE_KEYWORDS) else 0.1
    up = sum(w in text_n for w in UP_WORDS)
    down = sum(w in text_n for w in DOWN_WORDS)
    sentiment = 0.0 if up == down else (1.0 if up > down else -1.0)
    return relevance, sentiment

def analyze_news_df(news_df, use_llm=True, progress_every=10):
    """Adds 'relevance' and 'sentiment_score' columns to news_df, using the LLM
    where available/successful and the keyword heuristic as fallback per-item."""
    if news_df.empty:
        news_df = news_df.copy()
        news_df["relevance"] = []
        news_df["sentiment_score"] = []
        news_df["weighted_sentiment"] = []
        return news_df

    client = get_anthropic_client() if use_llm else None
    cache = load_llm_cache()
    relevances, sentiments, sources = [], [], []

    for i, text in enumerate(news_df["text"]):
        llm_result = analyze_news_with_llm(client, text, cache) if client else None
        if llm_result and llm_result["relevance"] is not None:
            relevances.append(llm_result["relevance"])
            sentiments.append(llm_result["sentiment"])
            sources.append("llm")
        else:
            r, s = keyword_relevance_sentiment(text)
            relevances.append(r)
            sentiments.append(s)
            sources.append("keyword")
        if client and (i + 1) % progress_every == 0:
            save_llm_cache(cache)
            print(f"  scored {i + 1}/{len(news_df)}...")

    if client:
        save_llm_cache(cache)

    news_df = news_df.copy()
    news_df["relevance"] = relevances
    news_df["sentiment_score"] = sentiments
    news_df["score_source"] = sources
    news_df["weighted_sentiment"] = news_df["relevance"] * news_df["sentiment_score"]
    return news_df

if len(news_real_df):
    print("Analyzing real news data...")
    news_real_df = analyze_news_df(news_real_df)
    print(f"\nScoring source breakdown: {news_real_df['score_source'].value_counts().to_dict()}")
    print(f"Mean relevance: {news_real_df['relevance'].mean():.3f}  "
          f"(near 0 confirms this news source has little/no direct LYD relevance)")
    print(news_real_df[["timestamp", "relevance", "sentiment_score", "text"]].head(5))
else:
    print("No real news data to analyze yet (see Section 2).")


Analyzing real news data...
ANTHROPIC_API_KEY not set - falling back to keyword heuristic. Set the env var and re-run this cell to enable LLM scoring.

Scoring source breakdown: {'keyword': 40}
Mean relevance: 0.100  (near 0 confirms this news source has little/no direct LYD relevance)
            timestamp  relevance  sentiment_score                                               text
0 2026-08-16 19:40:00        0.1              0.0  ‏وزير الخارجية يستقبل نظيرته اليمنية ويستعرضان...
1 2026-08-16 23:19:00        0.1              0.0  الرئيس الأميركي:\n- نهنئ السعودية وتركيا وباكس...
2 2026-08-17 15:11:00        0.1              0.0  ‏وزير التعليم: الموافقة على نظام التعليم العام...
3 2026-08-17 15:13:00        0.1              0.0  ‏وزير التعليم: نعمل مع "سدايا" و"هيوماين" للعم...
4 2026-08-17 15:14:00        0.1              0.0  ‏وزير التعليم: دربنا 4000 معلم ومعلمة لاكتشاف ...


## 4. (Optional) Live WhatsApp Channel ingestion — for ongoing collection

Sections 2-3 bootstrap from a one-time chat export. This section is for keeping data flowing afterward without re-exporting by hand each time - connects live to the same two WhatsApp *Channels* (not the chat export above) via Baileys. Optional: skip straight to Section 5 if you'd rather just re-export the chat periodically instead.

Writes the ingestion script to disk (as a real `.js` file, so it can actually run as
Node.js) and installs its dependencies. This uses **Baileys**, an unofficial WhatsApp
Web client library — every API call it makes (`newsletterMetadata`, `newsletterFollow`,
`newsletterFetchMessages`, `isJidNewsletter`) was verified against the installed
package's source, not guessed. Two honest caveats carried over from building this:

- It authenticates as a real WhatsApp account (you'll scan a QR code below). Use a
  number you're comfortable dedicating to this — unofficial automation clients carry
  some risk of the account being flagged, though reading public channel content
  (no sending, no interacting with anyone) is about as low-risk as that gets.
- Historical backfill (`history` mode) is best-effort: the endpoint exists and is
  callable, but its raw response format is version-sensitive and wasn't verified
  against a live server. It dumps the raw response to JSON for inspection instead of
  guessing at parsing it, rather than presenting unverified parsing as reliable.


In [21]:
import subprocess, shutil, os, time, json
from pathlib import Path

INGEST_DIR = Path("whatsapp_ingest")
INGEST_DIR.mkdir(exist_ok=True)

RATES_CHANNEL_INVITE = "0029Vb6ANHP0wajrIcDQmi3w"  # >>> REPLACE if your channel link changes
NEWS_CHANNEL_INVITE = "0029VaD7VDCIt5s3Js6Wx50E"   # >>> REPLACE if your channel link changes

package_json = {
    "name": "lyd-whatsapp-ingest",
    "version": "1.0.0",
    "private": True,
    "type": "module",
    "dependencies": {
        "@whiskeysockets/baileys": "^7.0.0-rc14",
        "@hapi/boom": "^10.0.1",
        "pino": "^9.0.0",
        "qrcode-terminal": "^0.12.0",
    },
}
(INGEST_DIR / "package.json").write_text(json.dumps(package_json, indent=2))

INGEST_JS = '/**\n * WhatsApp Channel ingestion for the LYD forecast pipeline.\n *\n * Reads two WhatsApp Channels (WhatsApp calls these "newsletters" internally):\n *   - RATES_CHANNEL: posts that contain rate numbers\n *   - NEWS_CHANNEL:  posts that are market commentary/news\n * and writes them to CSVs matching the schema the forecasting notebook expects\n * (rates_from_whatsapp.csv, news_from_whatsapp.csv).\n *\n * Built on @whiskeysockets/baileys (unofficial WhatsApp Web client library).\n * Every API call below (newsletterMetadata, newsletterFollow, newsletterFetchMessages,\n * isJidNewsletter) was verified against the installed package source, not guessed.\n *\n * ---------------------------------------------------------------------------\n * IMPORTANT - READ BEFORE RUNNING\n * ---------------------------------------------------------------------------\n * 1. This is an UNOFFICIAL client library, not WhatsApp\'s official Business API.\n *    It works by authenticating as a real WhatsApp account (you\'ll scan a QR\n *    code). Use a number you\'re comfortable dedicating to this - unofficial\n *    automation clients carry some risk of the account being flagged/banned.\n *    Reading public channel content (this script doesn\'t send messages or\n *    interact with anyone) is about as low-risk as unofficial automation gets,\n *    but it\'s still outside WhatsApp\'s official ToS for automation.\n *\n * 2. TWO INGESTION MODES:\n *    - `node ingest.js live`    -> connects and saves NEW posts as they arrive.\n *      This path uses the same message pipeline as normal chats and is solid.\n *    - `node ingest.js history` -> attempts to pull past posts via\n *      newsletterFetchMessages. This endpoint\'s response format is more\n *      version-sensitive and I could not test it against a live WhatsApp\n *      server from this environment. Run `node ingest.js history --debug`\n *      first: it will save the raw server response to raw_history_debug.json\n *      instead of guessing at parsing it, so you can inspect the actual shape\n *      and I can help fix the parser against real data if it doesn\'t match.\n *\n * 3. Run `npm install` in this folder first (@whiskeysockets/baileys, pino).\n */\n\nimport makeWASocket, {\n  useMultiFileAuthState,\n  fetchLatestBaileysVersion,\n  isJidNewsletter,\n  DisconnectReason,\n} from "@whiskeysockets/baileys";\nimport { Boom } from "@hapi/boom";\nimport pino from "pino";\nimport qrcodeTerminal from "qrcode-terminal";\nimport fs from "fs";\nimport path from "path";\nimport { fileURLToPath } from "url";\n\nconst __dirname = path.dirname(fileURLToPath(import.meta.url));\n\n// ---------------------------------------------------------------------------\n// Config - the two channels from the user\'s links\n// ---------------------------------------------------------------------------\n// Invite code is just the part after "https://whatsapp.com/channel/"\nconst RATES_CHANNEL_INVITE = "0029Vb6ANHP0wajrIcDQmi3w";\nconst NEWS_CHANNEL_INVITE = "0029VaD7VDCIt5s3Js6Wx50E";\n\nconst RATES_CSV = path.join(__dirname, "rates_from_whatsapp.csv");\nconst NEWS_CSV = path.join(__dirname, "news_from_whatsapp.csv");\nconst AUTH_DIR = path.join(__dirname, "auth_state");\n\nconst logger = pino({ level: "warn" });\n\n// ---------------------------------------------------------------------------\n// CSV helpers - append-only, matching the notebook\'s expected schema\n// ---------------------------------------------------------------------------\nfunction ensureCsvHeader(filePath, header) {\n  if (!fs.existsSync(filePath)) {\n    fs.writeFileSync(filePath, header + "\\n");\n  }\n}\n\nfunction csvEscape(value) {\n  const s = String(value ?? "");\n  if (s.includes(",") || s.includes(\'"\') || s.includes("\\n")) {\n    return \'"\' + s.replace(/"/g, \'""\') + \'"\';\n  }\n  return s;\n}\n\nfunction appendRatesRow({ timestamp, pair, sellRate, buyRate, sourceGroup }) {\n  ensureCsvHeader(RATES_CSV, "timestamp,pair,sell_rate,buy_rate,source_group");\n  const row = [timestamp, pair, sellRate ?? "", buyRate ?? "", csvEscape(sourceGroup)].join(",");\n  fs.appendFileSync(RATES_CSV, row + "\\n");\n}\n\nfunction appendNewsRow({ timestamp, sourceGroup, text }) {\n  ensureCsvHeader(NEWS_CSV, "timestamp,source_group,text");\n  const row = [timestamp, csvEscape(sourceGroup), csvEscape(text)].join(",");\n  fs.appendFileSync(NEWS_CSV, row + "\\n");\n}\n\n// ---------------------------------------------------------------------------\n// Text parsing - same approach as the notebook\'s extract_mentioned_rate().\n// >>> ADJUST these once you\'ve seen real messages from the rates channel -\n// I haven\'t seen the channel\'s actual message format, so this is a\n// reasonable starting guess (a decimal number like "5.35"), not a\n// guaranteed match for however that specific channel formats its posts.\n// ---------------------------------------------------------------------------\nconst RATE_PATTERN = /(\\d{1,2}[.,]\\d{1,3})/g;\n\nfunction extractRates(text) {\n  if (!text) return [];\n  const matches = [...text.matchAll(RATE_PATTERN)]\n    .map((m) => parseFloat(m[1].replace(",", ".")))\n    .filter((v) => v > 1.0 && v < 15.0); // sane bounds for LYD pairs - adjust if needed\n  return matches;\n}\n\nfunction guessPair(text) {\n  if (!text) return "UNKNOWN";\n  if (text.includes("دولار") || /\\busd\\b/i.test(text)) return "USD/LYD";\n  if (text.includes("يورو") || /\\beur\\b/i.test(text)) return "EUR/LYD";\n  return "UNKNOWN"; // >>> ADJUST - add more currency keywords as needed\n}\n\nfunction extractMessageText(msg) {\n  const m = msg.message;\n  if (!m) return null;\n  return (\n    m.conversation ||\n    m.extendedTextMessage?.text ||\n    m.imageMessage?.caption ||\n    m.videoMessage?.caption ||\n    null\n  );\n}\n\nfunction messageTimestampIso(msg) {\n  const ts = msg.messageTimestamp;\n  const seconds = typeof ts === "number" ? ts : ts?.toNumber?.() ?? Number(ts);\n  return new Date(seconds * 1000).toISOString();\n}\n\n// ---------------------------------------------------------------------------\n// Connection\n//\n// IMPORTANT FIX (found by testing against a real WhatsApp account): right\n// after a QR scan, WhatsApp sends a "stream:error code 515" and force-closes\n// the socket - this is normal, expected first-time-pairing behavior, not a\n// failure. The old version of this script reconnected by calling connect()\n// again internally, which created a brand new socket that the caller\'s\n// business logic (follow channels, listen for messages) never attached to -\n// so it silently reconnected into a dead end and never did anything.\n//\n// Fix: a single session owns the current socket and re-fires onReady with\n// whatever socket is currently active every time the connection opens\n// (including after the expected post-pairing reconnect), so channel-follow\n// and the message listener always attach to a live socket.\n// ---------------------------------------------------------------------------\nasync function startSession({ durationSeconds = null, onReady }) {\n  const { state, saveCreds } = await useMultiFileAuthState(AUTH_DIR);\n\n  // FIX (found by testing): fetchLatestBaileysVersion() calls fetch() with NO\n  // timeout at all (confirmed in the installed package\'s source). In an\n  // environment with no real network path to GitHub/WhatsApp, this hangs\n  // indefinitely instead of failing - which is exactly the silent-freeze\n  // symptom to watch for. We race it against our own timeout and fall back to\n  // the version bundled with the installed package (same one the library\n  // itself falls back to on an explicit fetch error - we just also cover the\n  // "hangs forever" case, which the library doesn\'t).\n  const FALLBACK_VERSION = [2, 3000, 1043857760]; // bundled in this package\'s Defaults - update if you bump the baileys version\n  console.log("Checking latest WhatsApp Web version (10s timeout)...");\n  const version = await Promise.race([\n    fetchLatestBaileysVersion().then((r) => r.version),\n    new Promise((resolve) =>\n      setTimeout(() => {\n        console.log(\n          "Version check timed out after 10s (likely no network path to GitHub from here) " +\n            "- using bundled fallback version instead. This is fine; it doesn\'t need to be " +\n            "the exact latest version to work."\n        );\n        resolve(FALLBACK_VERSION);\n      }, 10000)\n    ),\n  ]);\n  console.log("Using WA version:", version);\n\n  let timerStarted = false;\n\n  function open() {\n    const sock = makeWASocket({\n      version,\n      logger,\n      auth: state,\n      connectTimeoutMs: 20000, // fail the actual WhatsApp connection fast too, instead of hanging\n      // NOTE: printQRInTerminal is deprecated as of Baileys 7.x (confirmed by\n      // testing against the installed package - it silently stopped showing\n      // the QR code). We render it ourselves from the \'qr\' field below.\n    });\n\n    // Diagnostic heartbeat: if nothing happens for a while, say so instead of\n    // going silent. This is exactly the symptom to watch for if you\'re in an\n    // environment with no real network path to WhatsApp\'s servers (e.g. some\n    // sandboxed/cloud notebook hosts) - the process just hangs with zero\n    // output until the overall timeout, which looks identical to "nothing is\n    // wrong, just slow" without this.\n    let sawAnyEvent = false;\n    const heartbeat = setInterval(() => {\n      if (!sawAnyEvent) {\n        console.log(\n          "...still waiting on WhatsApp (no event yet). If this repeats for a full " +\n            "session with no QR code and no error, this environment likely can\'t " +\n            "reach WhatsApp\'s servers at all - try running this from a normal " +\n            "machine/network instead. If you previously scanned a QR successfully " +\n            "and it\'s stuck resuming, delete the \'whatsapp_ingest/auth_state\' " +\n            "folder to force a fresh QR-based login."\n        );\n      }\n    }, 15000);\n\n    sock.ev.on("creds.update", saveCreds);\n\n    sock.ev.on("connection.update", async (update) => {\n      sawAnyEvent = true;\n      const { connection, lastDisconnect, qr } = update;\n      if (qr) {\n        console.log("\\nScan this QR code with WhatsApp (Linked Devices):\\n");\n        qrcodeTerminal.generate(qr, { small: true });\n      }\n      if (connection === "connecting") {\n        console.log("Connecting...");\n      } else if (connection === "close") {\n        clearInterval(heartbeat);\n        const statusCode = new Boom(lastDisconnect?.error)?.output?.statusCode;\n        const shouldReconnect = statusCode !== DisconnectReason.loggedOut;\n        console.log("Connection closed.", { statusCode, shouldReconnect });\n        if (shouldReconnect) {\n          console.log("Reconnecting (this is expected right after a QR scan)...");\n          open(); // re-run with a fresh socket; onReady fires again once it opens\n        }\n      } else if (connection === "open") {\n        clearInterval(heartbeat);\n        console.log("Connected to WhatsApp.");\n        try {\n          await onReady(sock); // always the CURRENTLY active socket, including after reconnects\n        } catch (err) {\n          console.error("onReady handler failed:", err);\n        }\n        if (durationSeconds && !timerStarted) {\n          timerStarted = true; // count the session window from first successful open, not from process start\n          setTimeout(() => {\n            console.log(`\\nReached ${durationSeconds}s time limit - closing connection.`);\n            try { sock.end(undefined); } catch (e) {}\n            process.exit(0);\n          }, durationSeconds * 1000);\n        }\n      }\n    });\n  }\n\n  open();\n}\n\n// ---------------------------------------------------------------------------\n// Resolve invite code -> newsletter JID, and follow it\n// (following is required to receive live updates for a channel you don\'t\n// already follow from your phone)\n// ---------------------------------------------------------------------------\nasync function resolveAndFollow(sock, inviteCode, label) {\n  const metadata = await sock.newsletterMetadata("invite", inviteCode);\n  if (!metadata) {\n    throw new Error(\n      `Could not resolve channel \'${label}\' from invite code \'${inviteCode}\'. ` +\n        `Double check the link is still valid and the account used to connect can view it.`\n    );\n  }\n  console.log(`[${label}] resolved: "${metadata.name}" (${metadata.id}), ${metadata.subscribers} subscribers`);\n  try {\n    await sock.newsletterFollow(metadata.id);\n    console.log(`[${label}] followed.`);\n  } catch (err) {\n    console.log(`[${label}] follow call failed (may already be followed):`, err.message);\n  }\n  return metadata.id; // jid, e.g. "1203xxxxxxxxxx@newsletter"\n}\n\n// ---------------------------------------------------------------------------\n// Mode: live - listen for new posts and append to CSV as they arrive\n// ---------------------------------------------------------------------------\nasync function runLive(durationSeconds = null) {\n  await startSession({\n    durationSeconds,\n    onReady: async (sock) => {\n      const ratesJid = await resolveAndFollow(sock, RATES_CHANNEL_INVITE, "rates-channel");\n      const newsJid = await resolveAndFollow(sock, NEWS_CHANNEL_INVITE, "news-channel");\n\n      console.log(\n        durationSeconds\n          ? `Listening for new channel posts for ${durationSeconds}s...`\n          : "Listening for new channel posts... (Ctrl+C to stop)"\n      );\n\n      // Attached fresh to whichever socket is currently active - necessary\n      // since a reconnect creates a new socket with its own event emitter.\n      sock.ev.on("messages.upsert", ({ messages }) => {\n        for (const msg of messages) {\n          const jid = msg.key?.remoteJid;\n          if (!jid || !isJidNewsletter(jid)) continue;\n\n          const text = extractMessageText(msg);\n          if (!text) continue;\n          const timestamp = messageTimestampIso(msg);\n\n          if (jid === ratesJid) {\n            const rates = extractRates(text);\n            const pair = guessPair(text);\n            if (rates.length > 0) {\n              appendRatesRow({\n                timestamp,\n                pair,\n                sellRate: rates[0],\n                buyRate: rates[1] ?? "",\n                sourceGroup: "WhatsApp Rates Channel",\n              });\n              console.log(`[rate] ${timestamp} ${pair} ${rates[0]}`);\n            } else {\n              console.log(`[rate-channel] no number found in: "${text}" - check RATE_PATTERN`);\n            }\n          } else if (jid === newsJid) {\n            appendNewsRow({ timestamp, sourceGroup: "WhatsApp News Channel", text });\n            console.log(`[news] ${timestamp} "${text.slice(0, 60)}..."`);\n          }\n        }\n      });\n    },\n  });\n}\n\n// ---------------------------------------------------------------------------\n// Mode: history - best-effort backfill of past posts\n// ---------------------------------------------------------------------------\nasync function runHistory({ debug = false, count = 100 } = {}) {\n  await startSession({\n    onReady: async (sock) => {\n      const ratesJid = await resolveAndFollow(sock, RATES_CHANNEL_INVITE, "rates-channel");\n      const newsJid = await resolveAndFollow(sock, NEWS_CHANNEL_INVITE, "news-channel");\n\n      for (const [jid, label] of [[ratesJid, "rates"], [newsJid, "news"]]) {\n        console.log(`Fetching last ${count} messages for ${label}...`);\n        const result = await sock.newsletterFetchMessages(jid, count);\n\n        if (debug) {\n          const outFile = path.join(__dirname, `raw_history_debug_${label}.json`);\n          fs.writeFileSync(outFile, JSON.stringify(result, null, 2));\n          console.log(\n            `[debug] saved raw response to ${outFile} - inspect it, then adjust ` +\n              `the parsing in runHistory() to match the real structure before relying on it.`\n          );\n          continue;\n        }\n\n        // Best-effort: the response is a raw XML-ish binary node. Its children\n        // are NOT guaranteed to be pre-decoded WAMessage objects across Baileys\n        // versions - this block is a starting point, not a verified parser.\n        // If this throws or comes back empty, run with --debug and inspect the\n        // JSON dump instead.\n        console.log(\n          `[${label}] raw result received (top-level keys: ${Object.keys(result || {}).join(", ")}). ` +\n            `Automatic parsing of history isn\'t verified in this environment - ` +\n            `run with --debug to inspect the shape, or rely on \'live\' mode ` +\n            `(fully verified) and let it accumulate data going forward.`\n        );\n      }\n\n      process.exit(0);\n    },\n  });\n}\n\n// ---------------------------------------------------------------------------\n// CLI entry\n// ---------------------------------------------------------------------------\nconst mode = process.argv[2];\nconst debugFlag = process.argv.includes("--debug");\n\nif (mode === "live") {\n  const durationArg = process.argv[3];\n  const duration = durationArg ? parseInt(durationArg, 10) : null;\n  runLive(duration);\n} else if (mode === "history") {\n  runHistory({ debug: debugFlag });\n} else {\n  console.log("Usage:");\n  console.log("  node ingest.js live              # runs until Ctrl+C - reliable, accumulates data going forward");\n  console.log("  node ingest.js live 90           # same, but auto-stops after 90s - fits a bounded notebook cell");\n  console.log("  node ingest.js history --debug   # best-effort backfill of past posts, inspect-first mode");\n  process.exit(1);\n}\n'

(INGEST_DIR / "ingest.js").write_text(INGEST_JS)
print("Wrote", INGEST_DIR / "package.json", "and", INGEST_DIR / "ingest.js")

node_ok = shutil.which("node") is not None
npm_ok = shutil.which("npm") is not None
print(f"node available: {node_ok}   npm available: {npm_ok}")

if not (node_ok and npm_ok):
    print("Node.js/npm not found in this environment - install Node.js 18+ first "
          "(e.g. `apt-get install -y nodejs npm` on Kaggle/Colab, or nodejs.org "
          "locally), then re-run this cell. Falling through - Section 4 will use "
          "synthetic data.")
else:
    install = subprocess.run(["npm", "install"], cwd=INGEST_DIR, capture_output=True, text=True, timeout=180)
    print(install.stdout[-2000:])
    if install.returncode != 0:
        print("npm install failed:")
        print(install.stderr[-2000:])
    else:
        print("Dependencies installed.")


Wrote whatsapp_ingest/package.json and whatsapp_ingest/ingest.js
node available: True   npm available: True

added 2 packages, and audited 78 packages in 1s

18 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

Dependencies installed.


### 4b. Connect and pull data (live)

**Run this cell interactively** (not as part of "Run All") so you can see and scan the
QR code while it's live. It streams the Node process's output in real time, follows
both channels, listens for `LISTEN_SECONDS` seconds, and writes/appends to
`whatsapp_ingest/rates_from_whatsapp.csv` and `whatsapp_ingest/news_from_whatsapp.csv`
in the schema Section 5 expects. Safe to re-run — running it again for another window
just appends more rows, so leaving it and re-running periodically is how you build up
real volume over the days (channels post a few times/day, not continuously).

If it can't connect (no Node, no network path to WhatsApp, or nothing scanned in
time), it fails without crashing the notebook — Section 5 will fall back to synthetic
data and clearly say so.


In [22]:
LISTEN_SECONDS = 90  # >>> increase for a longer window if you have time to wait for the QR scan

def run_streamed(cmd, cwd, timeout):
    """Stream a subprocess's output live so a QR code printed mid-run is visible
    in time to scan it. Enforces a REAL wall-clock deadline via select() rather
    than only checking between blocking line-reads - found by testing that the
    naive 'for line in proc.stdout' version hangs forever if the subprocess
    produces zero output (e.g. it's stuck on a network call with no timeout of
    its own), since the timeout check never gets a chance to run."""
    import select
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    start = time.time()
    try:
        while True:
            remaining = timeout - (time.time() - start)
            if remaining <= 0:
                print("\n[notebook] timeout reached, terminating...")
                proc.terminate()
                break
            ready, _, _ = select.select([proc.stdout], [], [], min(remaining, 1.0))
            if ready:
                line = proc.stdout.readline()
                if line == "":  # EOF - process finished
                    break
                print(line, end="")
            elif proc.poll() is not None:
                break  # process ended without more output
    except KeyboardInterrupt:
        proc.terminate()
    try:
        return proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        proc.kill()
        return -1

if len(rates_real_df) >= MIN_ROWS_TO_USE_REAL_DATA:
    print(f"Skipping live ingestion - Section 2 already found {len(rates_real_df)} real rate rows "
          f"from the chat export, which clears the {MIN_ROWS_TO_USE_REAL_DATA}-row bar. "
          f"Delete/rename rates.txt if you want to force live ingestion instead.")
elif node_ok and npm_ok:
    print(f"Connecting - scan the QR code below with WhatsApp (Linked Devices) within {LISTEN_SECONDS}s...\n")
    try:
        rc = run_streamed(["node", "ingest.js", "live", str(LISTEN_SECONDS)],
                           cwd=INGEST_DIR, timeout=LISTEN_SECONDS + 30)
        print(f"\n[notebook] ingest.js exited with code {rc}")
    except Exception as e:
        print(f"[notebook] ingestion failed: {e}")
else:
    print("Skipping - Node/npm unavailable (see previous cell).")

rates_wa_path = INGEST_DIR / "rates_from_whatsapp.csv"
news_wa_path = INGEST_DIR / "news_from_whatsapp.csv"
print("\nrates_from_whatsapp.csv exists:", rates_wa_path.exists(),
      "| rows:", sum(1 for _ in open(rates_wa_path)) - 1 if rates_wa_path.exists() else 0)
print("news_from_whatsapp.csv exists:", news_wa_path.exists(),
      "| rows:", sum(1 for _ in open(news_wa_path)) - 1 if news_wa_path.exists() else 0)


Skipping live ingestion - Section 2 already found 74 real rate rows from the chat export, which clears the 20-row bar. Delete/rename rates.txt if you want to force live ingestion instead.

rates_from_whatsapp.csv exists: False | rows: 0
news_from_whatsapp.csv exists: False | rows: 0


**Optional: one-shot history backfill.** Best-effort, debug-first (see the
caveat in Section 2) — run this if you want to try pulling past posts instead of only
new ones going forward. It saves raw responses to JSON rather than guessing at
parsing them.


In [23]:
if len(rates_real_df) >= MIN_ROWS_TO_USE_REAL_DATA:
    print(f"Skipping history backfill - Section 2 already found {len(rates_real_df)} real rate rows.")
elif node_ok and npm_ok:
    rc = run_streamed(["node", "ingest.js", "history", "--debug"], cwd=INGEST_DIR, timeout=60)
    print(f"\n[notebook] history dump exited with code {rc}")
    for label in ["rates", "news"]:
        p = INGEST_DIR / f"raw_history_debug_{label}.json"
        if p.exists():
            print(f"Saved {p} ({p.stat().st_size} bytes) - inspect it to help write a real parser.")


Skipping history backfill - Section 2 already found 74 real rate rows.


## 5. Data schema recap

Everything downstream expects two tables in this shape, regardless of which
source produced them (chat export, live ingestion, or synthetic fallback):

rates: timestamp, pair (e.g. "USD/LYD"), sell_rate, source_group
  (optional: direction, sharp_move -- present for real chat-export data, used
  as extra features if available)

news: timestamp, source_group, text, relevance, sentiment_score,
  weighted_sentiment (the last three added by Section 3)


## 6. Finalize the dataset

Priority order: (1) the real chat export parsed in Section 2 if it has enough
rows to be worth training on, (2) live-ingested WhatsApp Channel data from
Section 4 if that's what you ran instead, (3) synthetic demo data otherwise --
covering both USD/LYD and EUR/LYD so the manual-test section always has more
than one pair to try, and so the notebook is fully runnable even with nothing
else set up yet.


In [24]:
def generate_synthetic_data(pairs=("USD/LYD", "EUR/LYD"), n_days=240, seed=42):
    """Stand-in data generator so the notebook is runnable with zero real data.
    Mean-reverting random walk (NOT a plain random walk - an earlier version of
    this generator used an unbounded random-walk-of-a-random-walk and could
    drift to unrealistic values; found and fixed by testing).
    DELETE this whole function once you trust real data enough to not need it."""
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2025-01-01")
    pair_base = {"USD/LYD": 5.10, "EUR/LYD": 5.55}
    all_rates = []
    for pair in pairs:
        anchor = pair_base.get(pair, 5.0)
        base = anchor
        momentum = 0.0
        for d in range(n_days):
            day = start + pd.Timedelta(days=d)
            for h in sorted(rng.choice(range(7, 22), size=rng.integers(2, 6), replace=False)):
                momentum = 0.9 * momentum + rng.normal(0, 0.0015)
                base = base + 0.015 * (anchor - base) + momentum + rng.normal(0, 0.006)
                base = max(anchor * 0.5, base)
                all_rates.append({
                    "timestamp": day + pd.Timedelta(hours=int(h), minutes=int(rng.integers(0, 60))),
                    "pair": pair, "sell_rate": round(base, 4),
                })
    rates_df = pd.DataFrame(all_rates)
    rates_df["source_group"] = "synthetic"
    rates_df = rates_df.sort_values("timestamp").reset_index(drop=True)

    up = ["ارتفاع", "زيادة", "قفز", "صعود"]
    down = ["انخفاض", "تراجع", "هبوط", "نزول"]
    neutral = ["ثبات", "استقرار", "بدون تغيير"]
    currency_word = {"USD/LYD": "الدولار", "EUR/LYD": "اليورو"}
    rows = []
    for pair in pairs:
        word_for_pair = currency_word.get(pair, "العملة")
        for d in range(n_days):
            day = start + pd.Timedelta(days=d)
            for _ in range(rng.integers(0, 5)):
                word = rng.choice(up) if rng.random() < 0.35 else rng.choice(down) if rng.random() < 0.7 else rng.choice(neutral)
                ts = day + pd.Timedelta(hours=int(rng.integers(6, 23)), minutes=int(rng.integers(0, 60)))
                rows.append({"timestamp": ts, "source_group": "synthetic", "text": f"سعر {word_for_pair} اليوم {word}"})
    news_df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    news_df = analyze_news_df(news_df, use_llm=False)  # keyword path only for synthetic data - keeps it fast/free
    return rates_df, news_df


rates_wa_path = Path("whatsapp_ingest") / "rates_from_whatsapp.csv"
news_wa_path = Path("whatsapp_ingest") / "news_from_whatsapp.csv"
live_ingest_available = (
    rates_wa_path.exists() and sum(1 for _ in open(rates_wa_path)) - 1 >= MIN_ROWS_TO_USE_REAL_DATA
)

if len(rates_real_df) >= MIN_ROWS_TO_USE_REAL_DATA:
    print(f"Using REAL chat-export data from Section 2: {len(rates_real_df)} rate rows, {len(news_real_df)} news rows.")
    rates_df, news_df = rates_real_df, news_real_df
elif live_ingest_available:
    print("Using live-ingested WhatsApp Channel data from Section 4.")
    rates_df = pd.read_csv(rates_wa_path, parse_dates=["timestamp"])
    rates_df = rates_df[rates_df["pair"] != "UNKNOWN"]
    news_df = pd.read_csv(news_wa_path, parse_dates=["timestamp"]) if news_wa_path.exists() else pd.DataFrame()
    news_df = analyze_news_df(news_df, use_llm=False) if len(news_df) else news_df
else:
    print(f"Not enough real data yet (need >= {MIN_ROWS_TO_USE_REAL_DATA} rate rows) - using SYNTHETIC demo data. "
          f"Re-run Section 2 with real files, or Section 4 for live ingestion, to switch over automatically.")
    rates_df, news_df = generate_synthetic_data()

SUPPORTED_PAIRS = sorted(rates_df["pair"].unique().tolist())
print(f"\nSUPPORTED_PAIRS: {SUPPORTED_PAIRS}")
print(f"rates_df: {len(rates_df)} rows, {rates_df.timestamp.min()} -> {rates_df.timestamp.max()}")
if len(news_df):
    print(f"news_df: {len(news_df)} rows, {news_df.timestamp.min()} -> {news_df.timestamp.max()}")


Using REAL chat-export data from Section 2: 74 rate rows, 40 news rows.

SUPPORTED_PAIRS: ['EUR/LYD', 'GBP/LYD', 'SUKUK/LYD', 'USD/LYD']
rates_df: 74 rows, 2026-08-13 18:42:00 -> 2026-08-18 18:20:00
news_df: 40 rows, 2026-08-16 19:40:00 -> 2026-08-18 22:18:00


## 7. Build the hourly feature set for one pair

Rate reports and news arrive at irregular times: rates are forward-filled (the
last known rate holds until a new one is posted); news is aggregated per hour
(mean weighted-sentiment, mean relevance, message count); missing hours = 0 (no
news that hour, not "neutral news"). News isn't filtered per-currency here --
unlike an earlier version of this pipeline, your real news source doesn't
mention specific currencies at all (see Section 2's finding), so the same
aggregated signal applies to every pair, which is both simpler and more honest
than pretending to split it.

When real chat-export data is in use, two extra features come along for free:
`direction` (the channel's own self-reported up/down/flat arrow) and
`sharp_move` (its lightning-bolt emphasis marker) -- genuine signal from the
source itself, not just something we derive by differencing the rate.


In [25]:
def build_hourly_dataset(pair: str, rates_df: pd.DataFrame, news_df: pd.DataFrame) -> pd.DataFrame:
    pair_rates = rates_df[rates_df["pair"] == pair]
    if pair_rates.empty:
        raise ValueError(f"No rate data for pair '{pair}'. Available: {sorted(rates_df['pair'].unique())}")

    rates_h = pair_rates.set_index("timestamp")[["sell_rate"]].resample("h").mean()
    rates_h["sell_rate"] = rates_h["sell_rate"].ffill()
    rates_h = rates_h.dropna()

    if "direction" in pair_rates.columns:
        extra = pair_rates.set_index("timestamp")[["direction", "sharp_move"]].resample("h").last()
        extra["direction"] = extra["direction"].ffill().fillna(0)
        extra["sharp_move"] = extra["sharp_move"].astype(float).fillna(0)
        rates_h = rates_h.join(extra, how="left")
        rates_h["direction"] = rates_h["direction"].fillna(0)
        rates_h["sharp_move"] = rates_h["sharp_move"].fillna(0)

    if len(news_df) and "weighted_sentiment" in news_df.columns:
        news_h = news_df.set_index("timestamp").resample("h").agg(
            sentiment_mean=("weighted_sentiment", "mean"),
            relevance_mean=("relevance", "mean"),
            news_count=("text", "count"),
        )
    else:
        news_h = pd.DataFrame(index=rates_h.index, columns=["sentiment_mean", "relevance_mean", "news_count"])

    df = rates_h.join(news_h, how="left")
    df[["sentiment_mean", "relevance_mean", "news_count"]] = df[["sentiment_mean", "relevance_mean", "news_count"]].fillna(0)
    return df

# quick look at one pair
build_hourly_dataset(SUPPORTED_PAIRS[0], rates_df, news_df).tail()


,sell_rate,direction,sharp_move,sentiment_mean,relevance_mean,news_count
timestamp,,,,,,
2026-08-18 12:00:00,10.560,-1.0,1.0,0.0,0.0,0.0
2026-08-18 13:00:00,10.545,-1.0,0.0,0.0,0.1,2.0
2026-08-18 14:00:00,10.510,-1.0,0.0,0.0,0.0,0.0
2026-08-18 15:00:00,10.510,-1.0,0.0,0.0,0.1,8.0
2026-08-18 16:00:00,10.500,1.0,1.0,0.0,0.1,3.0


## 8. Feature engineering

Lag features + rolling stats on the rate itself, plus rolling news
sentiment/volume and (for real data) the self-reported direction/sharp-move
signals. `hour` and `dow` capture intraday/weekly patterns.


In [26]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for lag in [1, 3, 6, 12, 24, 48]:
        df[f"rate_lag_{lag}"] = df["sell_rate"].shift(lag)

    df["rate_roll_mean_24"] = df["sell_rate"].rolling(24).mean()
    df["rate_roll_std_24"] = df["sell_rate"].rolling(24).std()
    df["rate_change_24"] = df["sell_rate"] - df["sell_rate"].shift(24)
    df["rate_change_pct_24"] = df["rate_change_24"] / df["sell_rate"].shift(24)

    df["sent_roll_mean_24"] = df["sentiment_mean"].rolling(24).mean()
    df["sent_roll_mean_6"] = df["sentiment_mean"].rolling(6).mean()
    df["news_count_roll_24"] = df["news_count"].rolling(24).sum()
    df["relevance_roll_mean_24"] = df["relevance_mean"].rolling(24).mean()

    if "direction" in df.columns:
        df["direction_roll_mean_12"] = df["direction"].rolling(12).mean()
        df["sharp_move_roll_sum_24"] = df["sharp_move"].rolling(24).sum()

    df["hour"] = df.index.hour
    df["dow"] = df.index.dayofweek

    df["target_24h"] = df["sell_rate"].shift(-24)
    df["target_48h"] = df["sell_rate"].shift(-48)
    return df

FEATURE_COLS = None  # set after processing the first pair, reused for all pairs

def data_sufficiency_note(n_usable_rows):
    """A single honest sentence about how much to trust results at this sample size."""
    if n_usable_rows < 15:
        return "VERY LOW - treat any output as a pipeline smoke-test only, not a real forecast."
    elif n_usable_rows < 60:
        return "LOW - directionally interesting at best; keep collecting data before relying on this."
    elif n_usable_rows < 200:
        return "MODERATE - usable for cautious decisions, still worth more data."
    else:
        return "REASONABLE - enough history for the backtest numbers to mean something."


## 9. Train models for every supported pair

One LightGBM regressor per horizon (24h, 48h) for the point forecast, plus
quantile regressors (10th / 90th percentile) for an 80% confidence interval.
Looped across `SUPPORTED_PAIRS` so every pair gets its own models. Prints an
explicit data-sufficiency note per pair -- with real sample data this will
likely say LOW or VERY LOW, which is the honest state of a 5-day-old dataset,
not a bug.


In [27]:
def train_horizon(train, test, feature_cols, target_col):
    models = {}
    for name, params in {
        "point": dict(objective="regression"),
        "q10": dict(objective="quantile", alpha=0.10),
        "q90": dict(objective="quantile", alpha=0.90),
    }.items():
        m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                               verbosity=-1, **params)
        m.fit(train[feature_cols], train[target_col])
        models[name] = m
    return models

pair_artifacts = {}  # pair -> {"models_24h", "models_48h", "train", "test", "model_df"}

for pair in SUPPORTED_PAIRS:
    raw = build_hourly_dataset(pair, rates_df, news_df)
    df = engineer_features(raw)
    feature_cols = [c for c in df.columns if c not in ["sell_rate", "target_24h", "target_48h"]]
    if FEATURE_COLS is None:
        FEATURE_COLS = feature_cols
    # align columns across pairs (real data may have direction/sharp_move, synthetic won't)
    for col in FEATURE_COLS:
        if col not in df.columns:
            df[col] = 0
    model_df = df.dropna(subset=FEATURE_COLS + ["target_24h", "target_48h"])

    note = data_sufficiency_note(len(model_df))
    split_idx = max(1, int(len(model_df) * 0.8))
    train, test = model_df.iloc[:split_idx], model_df.iloc[split_idx:]

    if len(train) < 5 or len(test) < 2:
        print(f"{pair}: only {len(model_df)} usable rows ({note}) - too few to train/test meaningfully, skipping.")
        continue

    models_24h = train_horizon(train, test, FEATURE_COLS, "target_24h")
    models_48h = train_horizon(train, test, FEATURE_COLS, "target_48h")

    pair_artifacts[pair] = {
        "models_24h": models_24h, "models_48h": models_48h,
        "train": train, "test": test, "model_df": model_df,
    }
    print(f"{pair}: trained on {len(train)} rows, holding out {len(test)} rows "
          f"({model_df.index.min()} -> {model_df.index.max()}) - data sufficiency: {note}")

if not pair_artifacts:
    raise RuntimeError("No pair had enough data to train on - check Section 2/6 output above.")


EUR/LYD: trained on 18 rows, holding out 5 rows (2026-08-15 18:00:00 -> 2026-08-16 16:00:00) - data sufficiency: LOW - directionally interesting at best; keep collecting data before relying on this.
GBP/LYD: trained on 18 rows, holding out 5 rows (2026-08-15 18:00:00 -> 2026-08-16 16:00:00) - data sufficiency: LOW - directionally interesting at best; keep collecting data before relying on this.
SUKUK/LYD: trained on 20 rows, holding out 5 rows (2026-08-15 18:00:00 -> 2026-08-16 18:00:00) - data sufficiency: LOW - directionally interesting at best; keep collecting data before relying on this.
USD/LYD: trained on 20 rows, holding out 5 rows (2026-08-15 18:00:00 -> 2026-08-16 18:00:00) - data sufficiency: LOW - directionally interesting at best; keep collecting data before relying on this.


## 10. Testing & evaluation (backtest)

MAE, RMSE, MAPE, and directional accuracy for every pair x horizon, always
checked against the naive persistence baseline (predict: rate stays exactly
where it is right now). `beats_naive = False` means don't trust that pair/
horizon yet -- with a 5-day real dataset, expect to see this a lot; it's the
model correctly not overselling itself on too little history, not a defect in
the pipeline.


In [28]:
def directional_accuracy(current, actual, predicted):
    actual_dir = np.sign(actual.values - current.values)
    pred_dir = np.sign(predicted - current.values)
    return float(np.mean(actual_dir == pred_dir))

eval_rows = []
for pair, art in pair_artifacts.items():
    test = art["test"]
    for horizon_name, models, target_col in [("24h", art["models_24h"], "target_24h"),
                                              ("48h", art["models_48h"], "target_48h")]:
        preds = models["point"].predict(test[FEATURE_COLS])
        mae = mean_absolute_error(test[target_col], preds)
        rmse = mean_squared_error(test[target_col], preds) ** 0.5
        mape = mean_absolute_percentage_error(test[target_col], preds)
        naive_mae = mean_absolute_error(test[target_col], test["sell_rate"])
        dir_acc = directional_accuracy(test["sell_rate"], test[target_col], preds)
        eval_rows.append({
            "pair": pair, "horizon": horizon_name, "n_test_rows": len(test),
            "MAE": round(mae, 4), "RMSE": round(rmse, 4), "MAPE": f"{mape:.3%}",
            "naive_MAE": round(naive_mae, 4), "beats_naive": mae < naive_mae,
            "directional_accuracy": f"{dir_acc:.1%}",
            "data_sufficiency": data_sufficiency_note(len(art["model_df"])).split(" - ")[0],
        })

eval_df = pd.DataFrame(eval_rows)
eval_df


,pair,horizon,n_test_rows,MAE,RMSE,MAPE,naive_MAE,beats_naive,directional_accuracy,data_sufficiency
0,EUR/LYD,24h,5,0.0336,0.0480,0.318%,0.078,True,100.0%,LOW
1,EUR/LYD,48h,5,0.0450,0.0506,0.428%,0.075,True,100.0%,LOW
2,GBP/LYD,24h,5,0.0200,0.0316,0.165%,0.050,True,60.0%,LOW
3,GBP/LYD,48h,5,0.0400,0.0548,0.333%,0.040,False,80.0%,LOW
4,SUKUK/LYD,24h,5,0.0586,0.0620,0.632%,0.086,True,80.0%,LOW
5,SUKUK/LYD,48h,5,0.0500,0.0523,0.542%,0.056,True,100.0%,LOW
6,USD/LYD,24h,5,0.0344,0.0374,0.377%,0.066,True,100.0%,LOW
7,USD/LYD,48h,5,0.0473,0.0503,0.520%,0.030,False,100.0%,LOW


**Reading this table with a small real dataset:** `n_test_rows` and
`data_sufficiency` are there deliberately -- a MAE that "beats naive" on 3-5
test rows isn't meaningful yet, it's a coin flip with extra steps. Treat this
table as a pipeline correctness check until `data_sufficiency` reads MODERATE
or better for the pairs you care about.


## 11. Manual testing -- enter a currency, get a prediction

Pick a pair, get the 24h/48h point forecast plus an 80% confidence interval
and a plain-language confidence label. The confidence label compares this
forecast's interval width against that pair's typical width from backtesting
-- with only a handful of backtest rows right now, treat the label itself as
low-confidence-in-its-own-right until more data accumulates (the
`data_sufficiency` figure from Section 10 is the more honest signal for now).


In [29]:
DEGENERATE_WIDTH_PCT_THRESHOLD = 0.02  # below this, treat the interval as overfit noise, not real precision

def confidence_label(ci_width_pct: float, typical_width_pct: float) -> str:
    if ci_width_pct < DEGENERATE_WIDTH_PCT_THRESHOLD:
        return "Unreliable (overfit)"  # see note below - a near-zero interval on tiny data means the
                                        # model memorized specific points, not that it's precise
    if typical_width_pct <= 0:
        return "Unknown"
    ratio = ci_width_pct / typical_width_pct
    if ratio <= 0.85:
        return "High"
    elif ratio <= 1.3:
        return "Medium"
    else:
        return "Low"

typical_widths = {}
for pair, art in pair_artifacts.items():
    test = art["test"]
    typical_widths[pair] = {}
    for horizon_name, models in [("24h", art["models_24h"]), ("48h", art["models_48h"])]:
        low = models["q10"].predict(test[FEATURE_COLS])
        high = models["q90"].predict(test[FEATURE_COLS])
        widths_pct = np.abs(high - low) / test["sell_rate"].values * 100
        typical_widths[pair][horizon_name] = float(np.median(widths_pct)) if len(widths_pct) else 0.0


def predict_rate(pair: str, as_of=None) -> dict:
    """The main manual-test entry point.
    pair: e.g. "USD/LYD" - must be one of pair_artifacts.keys()
    as_of: optional historical timestamp to test against (defaults to latest available data)
    """
    pair = pair.strip().upper().replace(" ", "")
    if pair not in pair_artifacts:
        raise ValueError(f"'{pair}' not supported. Trained pairs: {list(pair_artifacts.keys())}")

    art = pair_artifacts[pair]
    model_df = art["model_df"]
    row = model_df.loc[[as_of]] if as_of is not None else model_df.iloc[[-1]]

    result = {"pair": pair, "as_of": str(row.index[0]), "current_rate": float(row["sell_rate"].iloc[0])}
    for horizon_name, models in [("24h", art["models_24h"]), ("48h", art["models_48h"])]:
        point = float(models["point"].predict(row[FEATURE_COLS])[0])
        low = float(models["q10"].predict(row[FEATURE_COLS])[0])
        high = float(models["q90"].predict(row[FEATURE_COLS])[0])
        low, high = min(low, high), max(low, high)
        ci_width_pct = (high - low) / point * 100 if point else 0
        label = confidence_label(ci_width_pct, typical_widths[pair][horizon_name])
        result[horizon_name] = {
            "point_forecast": round(point, 4),
            "confidence_interval_80pct": [round(low, 4), round(high, 4)],
            "confidence": label,
        }
    return result


def print_prediction(pair: str, as_of=None):
    r = predict_rate(pair, as_of)
    print(f"Pair: {r['pair']}   as of {r['as_of']}   current rate: {r['current_rate']}")
    for horizon in ["24h", "48h"]:
        h = r[horizon]
        print(f"  +{horizon}: {h['point_forecast']}  "
              f"(80% CI: {h['confidence_interval_80pct'][0]} - {h['confidence_interval_80pct'][1]})  "
              f"confidence: {h['confidence']}")
    if any(r[h]["confidence"] == "Unreliable (overfit)" for h in ["24h", "48h"]):
        n_train = len(pair_artifacts[r["pair"]]["train"])
        print(f"  Note: a near-zero-width interval here isn't real precision - with only "
              f"{n_train} training rows, the model likely memorized specific historical "
              f"points rather than learning genuine uncertainty. Treat the point forecast "
              f"as a rough estimate only until more data accumulates.")


### Try it

Edit `pair_to_test` below to any of the trained pairs (printed by the cell)
and re-run.


In [30]:
pair_to_test = SUPPORTED_PAIRS[0]  # >>> CHANGE THIS to test a different currency, e.g. "EUR/LYD"
print("Trained pairs:", list(pair_artifacts.keys()))
print_prediction(pair_to_test)


Trained pairs: ['EUR/LYD', 'GBP/LYD', 'SUKUK/LYD', 'USD/LYD']
Pair: EUR/LYD   as of 2026-08-16 16:00:00   current rate: 10.43
  +24h: 10.4944  (80% CI: 10.495 - 10.495)  confidence: Unreliable (overfit)
  +48h: 10.57  (80% CI: 10.57 - 10.57)  confidence: Unreliable (overfit)
  Note: a near-zero-width interval here isn't real precision - with only 18 training rows, the model likely memorized specific historical points rather than learning genuine uncertainty. Treat the point forecast as a rough estimate only until more data accumulates.


### Optional: interactive prompt

Same thing, but asks you to type the currency instead of editing a variable.
Falls back to a default automatically if run non-interactively (e.g. Kaggle's
"Save & Run All") instead of hanging on the prompt.


In [31]:
if sys.stdin.isatty():
    user_pair = input(f"Enter a currency pair {list(pair_artifacts.keys())}: ").strip()
else:
    user_pair = list(pair_artifacts.keys())[0]
    print(f"Non-interactive run detected - defaulting to {user_pair}. "
          f"Run this cell in an interactive session to type your own.")

try:
    print_prediction(user_pair)
except ValueError as e:
    print("Error:", e)


Non-interactive run detected - defaulting to EUR/LYD. Run this cell in an interactive session to type your own.
Pair: EUR/LYD   as of 2026-08-16 16:00:00   current rate: 10.43
  +24h: 10.4944  (80% CI: 10.495 - 10.495)  confidence: Unreliable (overfit)
  +48h: 10.57  (80% CI: 10.57 - 10.57)  confidence: Unreliable (overfit)
  Note: a near-zero-width interval here isn't real precision - with only 18 training rows, the model likely memorized specific historical points rather than learning genuine uncertainty. Treat the point forecast as a rough estimate only until more data accumulates.


## 12. Save model artifacts

In [32]:
joblib.dump({
    "pair_artifacts": {
        pair: {"models_24h": art["models_24h"], "models_48h": art["models_48h"]}
        for pair, art in pair_artifacts.items()
    },
    "feature_cols": FEATURE_COLS,
    "typical_widths": typical_widths,
    "supported_pairs": list(pair_artifacts.keys()),
}, "lyd_forecast_model.joblib")
print("Saved lyd_forecast_model.joblib")


Saved lyd_forecast_model.joblib


## 13. Serving this as an API for the Node backend

Not runnable on Kaggle (no network), but this is the FastAPI wrapper to deploy
the saved `.joblib` behind, mirroring `predict_rate()` above so behavior
matches what you validated in Section 11.

```python
# forecast_service.py  (separate Python microservice, e.g. FastAPI + uvicorn)
from fastapi import FastAPI, HTTPException
import joblib
import pandas as pd

app = FastAPI()
artifact = joblib.load("lyd_forecast_model.joblib")
FEATURE_COLS = artifact["feature_cols"]

@app.post("/forecast")
def forecast(pair: str, features: dict):
    """features: dict of the engineered feature columns for the latest hour,
    computed by a shared feature-pipeline module (not duplicated in Node) -
    same columns as FEATURE_COLS, built the same way as build_hourly_dataset()
    + engineer_features() above."""
    if pair not in artifact["pair_artifacts"]:
        raise HTTPException(404, f"Unsupported pair \'{pair}\'")
    art = artifact["pair_artifacts"][pair]
    row = pd.DataFrame([features])[FEATURE_COLS]
    result = {"pair": pair, "generated_at": pd.Timestamp.utcnow().isoformat()}
    for horizon, models in [("24h", art["models_24h"]), ("48h", art["models_48h"])]:
        point = float(models["point"].predict(row)[0])
        low, high = float(models["q10"].predict(row)[0]), float(models["q90"].predict(row)[0])
        result[horizon] = {
            "point_forecast": point,
            "confidence_interval_80pct": [min(low, high), max(low, high)],
        }
    return result
```


## 14. Next steps / TODO before this is production-grade

- [ ] Keep exporting `rates.txt` weekly (or set up Section 4's live ingestion)
      and re-running -- accuracy should improve materially as the ~5-day
      sample grows into weeks/months
- [ ] Connect a genuinely Libya/economy-focused news source -- the sample
      `news.txt` scored near-zero relevance across the board, which Section 3
      handled correctly, but a relevant source would actually help forecasts
- [ ] Don't trust `beats_naive = True` on small `n_test_rows` -- re-check
      Section 10 after every retrain as data grows, not just once
- [ ] Set `ANTHROPIC_API_KEY` to get real LLM-based news scoring instead of
      the keyword fallback (Section 3)
- [ ] Recalibrate `typical_widths` (Section 11) after every retrain
- [ ] Log every forecast + the eventual actual rate to measure live accuracy
      over time, not just backtest accuracy
- [ ] Retrain on a schedule (daily/weekly) once you have a reliable
      recurring data source
